In [11]:
library("R.matlab")
library("tidyverse")
library("afex")
library("BayesFactor")

In [12]:
extract_metrics <- function(filepath, group) {
  mat <- readMat(filepath)
  data.frame(
    Subject = basename(filepath),
    Group = group,
    Block = c('baseline', 'early_learning', 'late_learning'),
    meanRT = as.numeric(mat$meanRT[1:3]),
    meanMT = as.numeric(mat$meanMT[1:3])
  )
}

adult_files <- list.files('adult_data', pattern = '\\_Final_Results.mat$', full.names = TRUE)
child_files <- list.files('children_data', pattern = '\\_Final_Results.mat$', full.names = TRUE)

data_adult <- map_dfr(adult_files, ~extract_metrics(.x, 'adult'))
data_child <- map_dfr(child_files, ~extract_metrics(.x, 'child'))

data_all <- bind_rows(data_adult, data_child)

head(data_all)

,Subject,Group,Block,meanRT,meanMT
,<chr>,<chr>,<chr>,<dbl>,<dbl>
1,VML_MEG_011_Final_Results.mat,adult,baseline,0.3590,1.018000
2,VML_MEG_011_Final_Results.mat,adult,early_learning,0.3308,1.136467
3,VML_MEG_011_Final_Results.mat,adult,late_learning,0.3260,1.093533
4,VML_MEG_012_2_Final_Results.mat,adult,baseline,0.3590,1.018000
5,VML_MEG_012_2_Final_Results.mat,adult,early_learning,0.3308,1.136467
6,VML_MEG_012_2_Final_Results.mat,adult,late_learning,0.3260,1.093533


In [13]:
# response time anova
anova_rt <- aov_ez(
  id = "Subject",
  dv = "meanRT",
  data = data_all,
  between = "Group",
  within = "Block"
)

print(anova_rt)


Converting to factor: Group

Contrasts set to contr.sum for the following variables: Group



Anova Table (Type 3 tests)

Response: meanRT
       Effect          df  MSE       F   ges p.value
1       Group       1, 22 0.03 8.95 **  .270    .007
2       Block 1.62, 35.58 0.00    0.73  .003    .461
3 Group:Block 1.62, 35.58 0.00    0.03 <.001    .941
---
Signif. codes:  0 ‘***’ 0.001 ‘**’ 0.01 ‘*’ 0.05 ‘+’ 0.1 ‘ ’ 1

Sphericity correction method: GG 


In [14]:
# movement time anova
anova_mt <- aov_ez(
  id = "Subject",
  dv = "meanMT",
  data = data_all,
  between = "Group",
  within = "Block"
)

print(anova_mt)


Converting to factor: Group

Contrasts set to contr.sum for the following variables: Group



Anova Table (Type 3 tests)

Response: meanMT
       Effect          df  MSE         F  ges p.value
1       Group       1, 22 0.02      1.05 .033    .316
2       Block 1.33, 29.34 0.01 24.08 *** .239   <.001
3 Group:Block 1.33, 29.34 0.01      2.34 .030    .129
---
Signif. codes:  0 ‘***’ 0.001 ‘**’ 0.01 ‘*’ 0.05 ‘+’ 0.1 ‘ ’ 1

Sphericity correction method: GG 


In [15]:
#response time bayes factor
data_all$Subject <- as.factor(data_all$Subject)
data_all$Group <- as.factor(data_all$Group)
data_all$Block <- as.factor(data_all$Block)

bf_rt <- anovaBF(
  meanRT ~ Group * Block + Subject,
  data = data_all,
  whichRandom = "Subject"
)

print(bf_rt)

Bayes factor analysis
--------------
[1] Group + Subject                       : 5.203631  ±4.34%
[2] Block + Subject                       : 0.2120339 ±0.88%
[3] Group + Block + Subject               : 1.049399  ±2.94%
[4] Group + Block + Group:Block + Subject : 0.2221324 ±4.81%

Against denominator:
  meanRT ~ Subject 
---
Bayes factor type: BFlinearModel, JZS



In [16]:
#movement time bayes factor
bf_mt <- anovaBF(
  meanMT ~ Group * Block + Subject,
  data = data_all,
  whichRandom = "Subject"
)

print(bf_mt)

Bayes factor analysis
--------------
[1] Group + Subject                       : 0.5089765 ±1.63%
[2] Block + Subject                       : 161424.7  ±0.76%
[3] Group + Block + Subject               : 97445.16  ±1.28%
[4] Group + Block + Group:Block + Subject : 90006.42  ±2.84%

Against denominator:
  meanMT ~ Subject 
---
Bayes factor type: BFlinearModel, JZS



In [17]:
print(anova_mt)
print(anova_rt)


Anova Table (Type 3 tests)

Response: meanMT
       Effect          df  MSE         F  ges p.value
1       Group       1, 22 0.02      1.05 .033    .316
2       Block 1.33, 29.34 0.01 24.08 *** .239   <.001
3 Group:Block 1.33, 29.34 0.01      2.34 .030    .129
---
Signif. codes:  0 ‘***’ 0.001 ‘**’ 0.01 ‘*’ 0.05 ‘+’ 0.1 ‘ ’ 1

Sphericity correction method: GG 
Anova Table (Type 3 tests)

Response: meanRT
       Effect          df  MSE       F   ges p.value
1       Group       1, 22 0.03 8.95 **  .270    .007
2       Block 1.62, 35.58 0.00    0.73  .003    .461
3 Group:Block 1.62, 35.58 0.00    0.03 <.001    .941
---
Signif. codes:  0 ‘***’ 0.001 ‘**’ 0.01 ‘*’ 0.05 ‘+’ 0.1 ‘ ’ 1

Sphericity correction method: GG 


In [18]:
anova(lm(meanMT ~ Group * Block, data = data_all))

,Df,Sum Sq,Mean Sq,F value,Pr(>F)
,<int>,<dbl>,<dbl>,<dbl>,<dbl>
Group,1,0.02108027,0.021080275,2.252095,1.382027e-01
Block,2,0.20149263,0.100746315,10.763155,9.002753e-05
Group:Block,2,0.01886514,0.009432571,1.007722,3.705965e-01
Residuals,66,0.61777951,0.009360296,NA,NA


In [19]:
anova(lm(meanRT ~ Group * Block, data = data_all))

,Df,Sum Sq,Mean Sq,F value,Pr(>F)
,<int>,<dbl>,<dbl>,<dbl>,<dbl>
Group,1,2.253172e-01,2.253172e-01,24.451176802,5.521157e-06
Block,2,1.868913e-03,9.344563e-04,0.101406189,9.037064e-01
Group:Block,2,8.593606e-05,4.296803e-05,0.004662844,9.953483e-01
Residuals,66,6.081889e-01,9.214983e-03,NA,NA
